# Аналіз поведінки користувачів Movie Match

## 1. Завантаження та первинне ознайомлення з даними (Data Ingestion)

In [4]:
# Імпортуємо необхідні бібліотеки
import pandas as pd
from pathlib import Path

In [6]:
# Визначаємо можливі шляхи до директорії з вихідними даними
possible_paths = [
    Path("data_raw"),
    Path("../data/simulated"),
    Path("data/simulated")
]

# Знаходимо першу директорію, яка містить CSV-файли
data_path = next(
    (
        path for path in possible_paths
        if path.is_dir() and any(path.glob("*.csv"))
    ),
    None
)

# Перевіряємо наявність директорії з даними
if data_path is None:
    raise FileNotFoundError(
        "Не знайдено директорію з CSV-файлами. "
        "Перевірте папку data_raw."
    )

print("Шлях до вихідних даних:", data_path.resolve())

Шлях до вихідних даних: C:\Users\ACER\github\analytics\data\simulated


In [7]:
# Отримуємо список CSV-файлів
csv_files = sorted(data_path.glob("*.csv"))

# Перевіряємо наявність файлів
if not csv_files:
    raise FileNotFoundError("У директорії немає CSV-файлів")

print(f"Кількість знайдених файлів: {len(csv_files)}")

Кількість знайдених файлів: 13


In [68]:
# Завантажуємо кожен CSV-файл в окремий DataFrame
# Зберігаємо всі таблиці у словнику datasets
datasets = {
    file.stem: pd.read_csv(file)
    for file in csv_files
}

# Перевіряємо розмір кожного завантаженого DataFrame
for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} рядків, {df.shape[1]} стовпців")

01_landing_page: 15000 рядків, 29 стовпців
02_authenticated_create_room: 900 рядків, 29 стовпців
02_guest_create_room: 1375 рядків, 29 стовпців
03_authenticated_host_room_invite_friends: 900 рядків, 29 стовпців
03_guest_host_room_invite_friends: 1100 рядків, 29 стовпців
06_authenticated_swipe_session: 16170 рядків, 29 стовпців
06_guest_swipe_session: 21230 рядків, 29 стовпців
08_authenticated_results_winners: 3813 рядків, 29 стовпців
08_guest_results_winners: 3903 рядків, 29 стовпців
09_sign_in: 960 рядків, 29 стовпців
10_sign_up: 636 рядків, 29 стовпців
11_home_signed_in: 3012 рядків, 29 стовпців
12_settings: 432 рядків, 29 стовпців


In [8]:
# Завантажуємо CSV-файли в окремі DataFrame
datasets = {
    file.stem: pd.read_csv(file)
    for file in csv_files
}

print("Дані успішно завантажено.")

Дані успішно завантажено.


In [10]:
# Формуємо зведену інформацію про завантажені датасети
dataset_summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    }
    for name, df in datasets.items()
])

display(dataset_summary)

,dataset,rows,columns
0,01_landing_page,15000,29
1,02_authenticated_create_room,900,29
2,02_guest_create_room,1375,29
3,03_authenticated_host_room_invite_friends,900,29
4,03_guest_host_room_invite_friends,1100,29
5,06_authenticated_swipe_session,16170,29
6,06_guest_swipe_session,21230,29
7,08_authenticated_results_winners,3813,29
8,08_guest_results_winners,3903,29
9,09_sign_in,960,29


У межах аналізу продуктових метрик було досліджено використання основних функцій Movie Match, налаштування ігрових кімнат, взаємодію зі сторінкою результатів, способи запрошення друзів та тривалість життєвого циклу кімнат.

**1. Основні продуктові показники (Product KPI)**

У досліджуваному наборі даних було зафіксовано 500 створених ігрових кімнат, з яких усі 500 (100%) мали подію запуску сесії.

На рівні учасників зафіксовано:
- 1 700 унікальних пар «кімната — учасник» із подією початку голосування;
- 1 700 пар із подією завершення голосування;
- 100% охоплення подією завершення серед пар, для яких зафіксовано початок сесії.

У 499 із 500 кімнат (99,8%) було зафіксовано подію `movie_won`. Таким чином, лише одна кімната не мала зареєстрованої події визначення переможця.

Отримані результати свідчать про високий рівень проходження основних етапів продуктового сценарію в досліджуваному наборі даних. Водночас кімнату без зафіксованого переможця доцільно додатково перевірити, щоб з'ясувати, чи пов'язано це з особливостями завершення процесу або відсутністю відповідної події в системі аналітики.

**2. Аналіз налаштувань ігрових кімнат (Room Configuration)**

Усього було проаналізовано 500 створених кімнат: 275 Guest та 225 Authenticated.

Найпопулярнішим типом контенту виявилися фільми (Movies):
- Guest — 214 кімнат (77,82%);
- Authenticated — 179 кімнат (79,56%).

Загалом фільми обрали у 393 кімнатах (78,6%), тоді як телевізійні серіали (TV Shows) — у 107 кімнатах (21,4%).

Розподіл обраної максимальної тривалості виявився досить рівномірним. Для Authenticated найчастіше обирали 120 хвилин (35,11%), а для Guest — 150 хвилин (34,55%). Значної переваги одного із запропонованих значень тривалості не спостерігається.

Окремо було досліджено вибір жанрів:
- У 388 кімнатах (77,6%) було зафіксовано вибір хоча б одного жанру.
- У 112 кімнатах (22,4%) жанри не обиралися.
- Середня кількість вибраних жанрів становить 1,18 для Authenticated та 1,21 для Guest.
- Медіанне значення в обох сегментах дорівнює одному жанру.

Серед найпопулярніших жанрів для Authenticated виділяються Western (10,22%), History і Mystery (по 8,89%), Documentary (8,44%) та Comedy (8,0%).

Для Guest найбільшу популярність мають Horror і Science Fiction (по 8,73%), Comedy (8,36%) та Mystery (8,0%).

Отже, користувачі переважно обирають фільми, а не серіали, при цьому налаштування тривалості мають досить рівномірний розподіл. Вибір жанрів зазвичай обмежується одним або кількома значеннями, а приблизно п'ята частина кімнат створюється без зазначення жанрових уподобань.

**3. Використання функцій сторінки результатів (Results Page Engagement)**

Аналіз взаємодії зі сторінкою результатів показав відмінності у використанні її додаткових функцій між Guest та Authenticated.

Серед авторизованих користувачів найбільш популярною функцією став перегляд деталей фільму-переможця (`winner_details_clicked`) — 43,43% відвідувачів сторінки.

Інші функції також демонструють помітний рівень використання:
- `view_sessions_clicked` — 39,14%;
- `explore_taste_clicked` — 37,0%;
- `another_round_clicked` — 18,23%.

У сегменті Guest найбільш використовуваною функцією також є перегляд деталей переможця — 22,69%. Повторний раунд (`another_round_clicked`) використовували 8,29% відвідувачів.

Для Guest у досліджуваних даних не зафіксовано подій `explore_taste_clicked` та `view_sessions_clicked`. Це може бути пов'язано з доступністю функцій для різних типів користувачів або особливостями відстеження подій, тому потребує окремої перевірки.

Таким чином, перегляд інформації про фільм-переможець є найбільш затребуваною з досліджених функцій сторінки результатів. Водночас серед авторизованих користувачів спостерігається ширше використання додаткових можливостей продукту.

**4. Аналіз використання функцій запрошення друзів (Invitation Features)**

Для залучення інших учасників до ігрової кімнати користувачі можуть копіювати посилання або використовувати запрошення через Telegram.

Копіювання посилання є найбільш популярним способом запрошення:
- Guest — 72,0% кімнат;
- Authenticated — 76,89% кімнат.

Запрошення через Telegram використовували значно рідше:
- Guest — 16,73% кімнат;
- Authenticated — 19,11% кімнат.

Загалом посилання було скопійовано у 371 із 500 кімнат (74,2%), тоді як запрошення через Telegram зафіксовано у 89 кімнатах (17,8%).

Отже, пряме посилання є основним способом запрошення друзів в обох сегментах. Це свідчить про важливість механізму копіювання та поширення посилань для організації спільного перегляду й голосування.

**5. Аналіз життєвого циклу ігрових кімнат (Room Lifecycle)**

Було проаналізовано проміжок часу від створення кімнати до завершення голосування останнім учасником.

Для всіх 500 створених кімнат вдалося встановити час завершення голосування. Пропущених значень тривалості та від'ємних значень не виявлено.

Основні результати:
- Середня тривалість життєвого циклу кімнати — 46,99 хвилини.
- Медіанна тривалість — 47,52 хвилини.
- Для Authenticated середня тривалість становить 47,16 хвилини.
- Для Guest — 46,85 хвилини.

Різниця між середніми значеннями для двох сегментів незначна — близько 0,31 хвилини.

Розподіл тривалості показує, що основна частина кімнат завершує голосування приблизно в межах 35–60 хвилин. Значного розходження між Guest та Authenticated за середньою тривалістю життєвого циклу не спостерігається.

Таким чином, у досліджуваних даних тривалість проходження основного ігрового сценарію є відносно стабільною та становить приблизно 47 хвилин.

**Загальний підсумок**

Результати аналізу продуктових метрик дозволили визначити основні особливості використання Movie Match:

- Усі 500 створених кімнат мали зафіксовану подію запуску сесії, а у 499 кімнатах було зареєстровано переможця.
- Найпопулярнішим типом контенту є Movies — 78,6% створених кімнат.
- Вибір максимальної тривалості кімнати розподілений досить рівномірно між варіантами 90, 120 та 150 хвилин.
- У 77,6% кімнат користувачі обрали хоча б один жанр, а середня кількість жанрів на кімнату становить приблизно 1,2.
- Перегляд деталей фільму-переможця є найбільш популярною з досліджених функцій сторінки результатів.
- Основним способом запрошення друзів є копіювання посилання, яке використовувалося приблизно у трьох чвертях кімнат.
- Середня тривалість життєвого циклу кімнати становить близько 47 хвилин і майже не відрізняється між Guest та Authenticated.

Отримані результати формують загальне розуміння використання основних можливостей продукту та дозволяють визначити напрями для подальшого дослідження: перевірку причин відсутності результату в окремій кімнаті, доступності функцій сторінки результатів для різних типів користувачів та ефективності механізмів запрошення друзів.

Наступним етапом є детальний аналіз користувацьких сценаріїв та воронок, який дозволить дослідити переходи між послідовними етапами взаємодії з продуктом.

Важливо враховувати, що дані є симульованими, тому отримані показники описують досліджуваний набір даних і не повинні автоматично трактуватися як фактичні показники реальної аудиторії Movie Match.